# EPEX Spot Market Analysis

This notebook provides exploratory data analysis (EDA) for EPEX Spot electricity market data.

## Note
EPEX Spot data is often accessible through the ENTSO-E Transparency Platform, as EPEX operates day-ahead markets for several European countries that report to ENTSO-E.

## Markets Covered
- Germany/Austria (DE-AT-LU)
- France (FR)
- Switzerland (CH)
- Belgium (BE)
- Netherlands (NL)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load EPEX Market Data

Load day-ahead price data from ENTSO-E for EPEX markets.

In [ ]:
# Define data directory
data_dir = Path('../data-scrapers/data')

# Function to load price data for different markets
def load_market_prices(country_code):
    """Load day-ahead prices for a specific market."""
    pattern = f'day_ahead_prices_{country_code}_*.csv'
    files = sorted(data_dir.glob(pattern))
    if files:
        print(f"Loading {country_code}: {files[-1].name}")
        df = pd.read_csv(files[-1], index_col=0, parse_dates=True)
        return df
    else:
        print(f"No files found for {country_code}")
        return None

# Load data for EPEX markets
markets = {
    'DE': 'Germany/Austria',
    'FR': 'France',
    'CH': 'Switzerland',
    'BE': 'Belgium',
    'NL': 'Netherlands'
}

market_data = {}
for code, name in markets.items():
    data = load_market_prices(code)
    if data is not None:
        market_data[name] = data

print(f"\nLoaded data for {len(market_data)} markets")

## 2. Multi-Market Price Comparison

In [ ]:
if market_data:
    # Combine all market data
    combined_prices = pd.DataFrame()
    for market_name, df in market_data.items():
        # Extract price column (handle both Series and DataFrame)
        if isinstance(df, pd.DataFrame):
            price_col = df.iloc[:, 0] if len(df.columns) > 0 else df
        else:
            price_col = df
        combined_prices[market_name] = price_col
    
    # Display summary statistics
    print("Price Statistics by Market (EUR/MWh):")
    print(combined_prices.describe())
    
    # Plot all markets together
    fig, ax = plt.subplots(figsize=(15, 8))
    combined_prices.plot(ax=ax, linewidth=1.5, alpha=0.8)
    ax.set_title('Day-Ahead Prices Across EPEX Markets', fontsize=16)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Price (EUR/MWh)', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Box plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    combined_prices.boxplot(ax=ax)
    ax.set_title('Price Distribution Comparison', fontsize=16)
    ax.set_ylabel('Price (EUR/MWh)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Correlation heatmap
    if len(combined_prices.columns) > 1:
        fig, ax = plt.subplots(figsize=(10, 8))
        correlation = combined_prices.corr()
        sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
                   center=0, square=True, linewidths=1)
        ax.set_title('Price Correlation Between Markets', fontsize=16)
        plt.tight_layout()
        plt.show()
        
        print("\nPrice Correlations:")
        print(correlation)

## 3. Market Coupling Analysis

Analyze price convergence and market coupling efficiency.

In [ ]:
if len(market_data) >= 2:
    # Calculate price spreads between markets
    if 'Germany/Austria' in combined_prices.columns and 'France' in combined_prices.columns:
        spread_de_fr = combined_prices['Germany/Austria'] - combined_prices['France']
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))
        
        # Plot prices
        combined_prices[['Germany/Austria', 'France']].plot(ax=ax1, linewidth=1.5)
        ax1.set_title('Germany/Austria vs France Prices', fontsize=14)
        ax1.set_ylabel('Price (EUR/MWh)', fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot spread
        spread_de_fr.plot(ax=ax2, linewidth=1.5, color='red')
        ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax2.set_title('Price Spread (DE-FR)', fontsize=14)
        ax2.set_xlabel('Date', fontsize=12)
        ax2.set_ylabel('Spread (EUR/MWh)', fontsize=12)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Spread statistics
        print("Price Spread Statistics (DE-FR):")
        print(f"Mean: {spread_de_fr.mean():.2f} EUR/MWh")
        print(f"Std Dev: {spread_de_fr.std():.2f} EUR/MWh")
        print(f"Min: {spread_de_fr.min():.2f} EUR/MWh")
        print(f"Max: {spread_de_fr.max():.2f} EUR/MWh")
        
        # Count hours with price convergence (spread < 1 EUR/MWh)
        convergence = (abs(spread_de_fr) < 1.0).sum()
        total_hours = len(spread_de_fr)
        convergence_pct = (convergence / total_hours) * 100
        print(f"\nPrice Convergence (spread < 1 EUR/MWh): {convergence_pct:.1f}% of hours")

## 4. Intraday Patterns Analysis

In [ ]:
if market_data:
    # Analyze hourly patterns for each market
    fig, axes = plt.subplots(len(market_data), 1, figsize=(12, 6*len(market_data)))
    
    if len(market_data) == 1:
        axes = [axes]
    
    for idx, (market_name, df) in enumerate(market_data.items()):
        # Extract price column
        if isinstance(df, pd.DataFrame):
            prices = df.iloc[:, 0] if len(df.columns) > 0 else df
        else:
            prices = df
        
        # Create DataFrame with hour information
        hourly_df = pd.DataFrame({'price': prices})
        hourly_df['hour'] = hourly_df.index.hour
        
        # Calculate statistics by hour
        hourly_stats = hourly_df.groupby('hour')['price'].agg(['mean', 'std', 'min', 'max'])
        
        # Plot
        ax = axes[idx]
        ax.plot(hourly_stats.index, hourly_stats['mean'], linewidth=2, label='Mean')
        ax.fill_between(hourly_stats.index,
                        hourly_stats['mean'] - hourly_stats['std'],
                        hourly_stats['mean'] + hourly_stats['std'],
                        alpha=0.3, label='±1 Std Dev')
        ax.set_title(f'Hourly Price Pattern - {market_name}', fontsize=14)
        ax.set_xlabel('Hour of Day', fontsize=12)
        ax.set_ylabel('Price (EUR/MWh)', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xticks(range(0, 24))
    
    plt.tight_layout()
    plt.show()

## 5. Volatility Analysis

In [ ]:
if market_data:
    # Calculate volatility metrics
    volatility_df = pd.DataFrame()
    
    for market_name, df in market_data.items():
        # Extract price column
        if isinstance(df, pd.DataFrame):
            prices = df.iloc[:, 0] if len(df.columns) > 0 else df
        else:
            prices = df
        
        # Calculate returns
        returns = prices.pct_change().dropna()
        
        # Calculate rolling volatility (24-hour window)
        rolling_vol = returns.rolling(window=24).std() * np.sqrt(24)  # Annualized
        
        volatility_df[market_name] = rolling_vol
    
    # Plot volatility
    fig, ax = plt.subplots(figsize=(15, 6))
    volatility_df.plot(ax=ax, linewidth=1.5)
    ax.set_title('Price Volatility (24h Rolling)', fontsize=16)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Volatility', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Average Volatility by Market:")
    print(volatility_df.mean().sort_values(ascending=False))

## 6. Export Analysis Results

In [ ]:
# Create summary report
if market_data:
    summary = {
        'markets_analyzed': list(market_data.keys()),
        'period': {
            'start': str(combined_prices.index.min()),
            'end': str(combined_prices.index.max())
        },
        'price_statistics': combined_prices.describe().to_dict(),
        'correlations': combined_prices.corr().to_dict() if len(combined_prices.columns) > 1 else {}
    }
    
    print("\nEPEX Market Analysis Summary:")
    print(f"Markets: {', '.join(summary['markets_analyzed'])}")
    print(f"Period: {summary['period']['start']} to {summary['period']['end']}")
    print("\nAverage prices:")
    for market in summary['markets_analyzed']:
        avg_price = combined_prices[market].mean()
        print(f"  {market}: {avg_price:.2f} EUR/MWh")

## Conclusions

Add your observations about EPEX Spot markets:
- Price levels and trends across markets
- Market coupling efficiency
- Volatility patterns
- Cross-border trade implications